# Dataset Preparation

In [ ]:
from datasets import load_dataset, Dataset

raw = load_dataset("DeepMount00/CulturaViva-ITA", split="train")
split_ds = raw.train_test_split(test_size=0.1, seed=42)

train_ds = split_ds["train"]
test_ds = split_ds["test"]


def build_ir_split(ds, split_name):
    queries = []
    corpus = []
    qrels = []

    for i, row in enumerate(ds):
        qid = f"{split_name}_q_{i}"
        did = f"{split_name}_d_{i}"

        queries.append(
            {
                "_id": qid,
                "text": row["prompt"],
            }
        )

        corpus.append(
            {
                "_id": did,
                "title": "",
                "text": row["answer"],
            }
        )

        qrels.append(
            {
                "query-id": qid,
                "corpus-id": did,
                "score": 1,
            }
        )

    return (
        Dataset.from_list(queries),
        Dataset.from_list(corpus),
        Dataset.from_list(qrels),
    )


queries_train, corpus_train, qrels_train = build_ir_split(train_ds, "train")
queries_test, corpus_test, qrels_test = build_ir_split(test_ds, "test")

repo_id = "lopozz/CulturaViva-Retrieval"

# push to repo
# DatasetDict({
#     "train": queries_train,
#     "test": queries_test,
# }).push_to_hub(repo_id, config_name="queries")

# # push corpus as one config
# DatasetDict({
#     "train": corpus_train,
#     "test": corpus_test,
# }).push_to_hub(repo_id, config_name="corpus")

# # push qrels as one config
# DatasetDict({
#     "train": qrels_train,
#     "test": qrels_test,
# }).push_to_hub(repo_id, config_name="qrels")

# Train

In [ ]:
from datasets import load_dataset

repo_id = "lopozz/CulturaViva-Retrieval"


def build_pair_dataset(
    repo_id: str, split: str, max_examples: int | None = None
) -> Dataset:
    queries_ds = load_dataset(repo_id, "queries", split=split)
    corpus_ds = load_dataset(repo_id, "corpus", split=split)
    qrels_ds = load_dataset(repo_id, "qrels", split=split)

    query_text_by_id = {row["_id"]: row["text"] for row in queries_ds}
    doc_text_by_id = {row["_id"]: row["text"] for row in corpus_ds}

    pairs = []
    for row in qrels_ds:
        if int(row["score"]) > 0:
            qid = row["query-id"]
            did = row["corpus-id"]

            if qid in query_text_by_id and did in doc_text_by_id:
                pairs.append(
                    {
                        "query": query_text_by_id[qid],
                        "answer": doc_text_by_id[did],
                    }
                )

                if max_examples is not None and len(pairs) >= max_examples:
                    break

    return Dataset.from_list(pairs)


train_dataset = build_pair_dataset(repo_id, "train", max_examples=100)
eval_dataset = build_pair_dataset(repo_id, "test", max_examples=10)

print(train_dataset)
print(eval_dataset)

In [ ]:
import logging

from sentence_transformers import (
    SparseEncoder,
    SparseEncoderModelCardData,
    SparseEncoderTrainer,
    SparseEncoderTrainingArguments,
)
from sentence_transformers.sentence_transformer.modules import Router, Transformer
from sentence_transformers.sparse_encoder.losses import (
    SparseMultipleNegativesRankingLoss,
    SpladeLoss,
)
from sentence_transformers.sparse_encoder.modules import (
    SparseStaticEmbedding,
    SpladePooling,
)
from sentence_transformers.sentence_transformer.training_args import BatchSamplers

logging.basicConfig(
    format="%(asctime)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S", level=logging.INFO
)

# 1. Load a model to finetune with 2. (Optional) model card data
mlm_transformer = Transformer("jhu-clsp/mmBERT-small", transformer_task="fill-mask")

splade_pooling = SpladePooling(
    pooling_strategy="max",
    embedding_dimension=mlm_transformer.get_embedding_dimension(),
)
router = Router.for_query_document(
    query_modules=[
        SparseStaticEmbedding(tokenizer=mlm_transformer.tokenizer, frozen=False)
    ],
    document_modules=[mlm_transformer, splade_pooling],
)

model = SparseEncoder(
    modules=[router],
    model_card_data=SparseEncoderModelCardData(
        language="it",
        license="apache-2.0",
        model_name="Inference-free SPLADE mmBERT-small trained on Natural-Questions tuples",
    ),
)


# 4. Define a loss function
loss = SpladeLoss(
    model=model,
    loss=SparseMultipleNegativesRankingLoss(model=model),
    query_regularizer_weight=0,
    document_regularizer_weight=3e-3,
)

batch_size = 1
# 5. (Optional) Specify training arguments
run_name = "inference-free-splade-mmBERT-small-ita"
args = SparseEncoderTrainingArguments(
    # Required parameter:
    output_dir=f"models/{run_name}",
    # Optional training parameters:
    num_train_epochs=1,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    learning_rate=2e-5,
    learning_rate_mapping={
        r"0\.sub_modules\.query\.0\.weight": 1e-3
    },  # Set a higher learning rate for the SparseStaticEmbedding module
    warmup_ratio=0.1,
    fp16=True,  # Set to False if you get an error that your GPU can't run on FP16
    bf16=False,  # Set to True if you have a GPU that supports BF16
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # MultipleNegativesRankingLoss benefits from no duplicate samples in a batch
    router_mapping={
        "query": "query",
        "answer": "document",
    },  # Map the column names to the routes
    # Optional tracking/debugging parameters:
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,
    logging_steps=200,
    report_to="none",
)

# 6. Create a trainer & train
trainer = SparseEncoderTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
)
trainer.train()

# 9. Save the trained model
model.save_pretrained(f"models/{run_name}/final")

# 10. (Optional) Push it to the Hugging Face Hub
# model.push_to_hub(run_name)

# Evaluate

In [ ]:
from datasets import load_dataset
from sentence_transformers import SparseEncoder, SparseEncoderModelCardData
from sentence_transformers.sparse_encoder.evaluation import (
    SparseInformationRetrievalEvaluator,
)
from sentence_transformers.sentence_transformer.modules import Router, Transformer
from sentence_transformers.sparse_encoder.modules import (
    SparseStaticEmbedding,
    SpladePooling,
)

# 1) Load Italian subsets
# params = {"path": "lopozz/CulturaViva-Retrieval", "split": "test"}
# names = ["corpus", "queries", "qrels"]
# params = {"path": "mteb/WikipediaRetrievalMultilingual", "split": "test"}
# names = ["it-corpus", "it-queries", "it-qrels"]
params = {"path": "mteb/WebFAQRetrieval", "split": "test"}
names = ["ita-corpus", "ita-queries", "ita-qrels"]
# params = {"path": "mteb/MuPLeR-retrieval", "split": "test"}
# names = ["it-corpus", "it-queries", "it-qrels"]
corpus_ds = load_dataset(name=names[0], **params)
queries_ds = load_dataset(name=names[1], **params)
qrels_ds = load_dataset(name=names[2], **params)

# 2) Convert queries to {query_id: text} and corpus to {corpus_id: text}
id = 'id'
# id = '_id'
queries = {row[id]: row["text"] for row in queries_ds}
corpus = {row[id]: row["text"] for row in corpus_ds}

# 3) Convert qrels to {query_id: set(doc_ids)}
# If you want, you can keep only positives with score > 0
relevant_docs = {}
for row in qrels_ds:
    qid = row["query-id"]
    cid = row["corpus-id"]
    score = row["score"]

    if score > 0:
        relevant_docs.setdefault(qid, set()).add(cid)

# 4) Build evaluator
ir_evaluator = SparseInformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    show_progress_bar=True,
    batch_size=4,
)

# 5) Load sparse model
# model_path = "../models/inference-free-splade-mmBERT-small-ita/final"
# model_path = "nickprock/splade-bert-base-italian-xxl-uncased-cv"
# model_path = "opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1"
model_path = "nickprock/splade-bert-base-italian-xxl-uncased-cv"
mlm_transformer = Transformer(model_path, transformer_task="fill-mask")

splade_pooling = SpladePooling(
    pooling_strategy="max",
    embedding_dimension=mlm_transformer.get_embedding_dimension(),
)
router = Router.for_query_document(
    # query_modules=[
    #     SparseStaticEmbedding(tokenizer=mlm_transformer.tokenizer, frozen=False)
    # ],
    query_modules=[mlm_transformer, splade_pooling],
    document_modules=[mlm_transformer, splade_pooling],
)

model = SparseEncoder(
    modules=[router],
    model_card_data=SparseEncoderModelCardData(
        language="it",
        license="apache-2.0",
        model_name="Inference-free SPLADE mmBERT-small trained on Natural-Questions tuples",
    ),
)

# 6) Run evaluation
results = ir_evaluator(model)
print(results)

## MTEB Benchmark

In [1]:
import yaml
import mteb

with open("../configs/ds/mteg-retrieval-ita.yml", "r") as f:
    config = yaml.safe_load(f)

excluded = set(config["excluded_tasks"])

tasks = mteb.get_tasks(
    languages=config["languages"],
    modalities=config["modalities"],
    task_types=config["task_types"],
    exclusive_language_filter=True,
    eval_splits=["test"],
)

tasks = [t for t in tasks if t.metadata.name not in excluded]

/home/lpozzi/Git/lm-toolkit/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import mteb

model_names = [
    "nickprock/splade-bert-base-italian-xxl-uncased-cv",
    "google/embeddinggemma-300m",
    "opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1"
]
cache = mteb.ResultCache("..")
results = cache.load_results(models=model_names, tasks=tasks, require_model_meta=False)

results.to_dataframe()

model_name,task_name,google/embeddinggemma-300m,nickprock/splade-bert-base-italian-xxl-uncased-cv,opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1
0,MuPLeR-retrieval,0.84454,0.51088,0.542586
1,WebFAQRetrieval,0.79523,NaN,0.222340
2,WikipediaRetrievalMultilingual,0.91761,0.79955,0.837200


In [4]:
import mteb
from sentence_transformers import SparseEncoder, SentenceTransformer

# model = SparseEncoder(
#     "opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1"
# )
cache = mteb.ResultCache("..")
model = SentenceTransformer(
    "opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1"
)

results = mteb.evaluate(
    model, tasks=tasks, cache=cache, encode_kwargs={"batch_size": 16}
)

Some weights of BertModel were not initialized from the model checkpoint at opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/lpozzi/Git/lm-toolkit/.venv/lib/python3.12/site-packages/mteb/models/model_meta.py:681: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dimensions = model.get_sentence_embedding_dimension()
/home/lpozzi/Git/lm-toolkit/.venv/lib/python3.12/site-packages/mteb/models/model_meta.py:646: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embed_dim=model.get_sentence_embedding_dimension(),
Evaluating task WebFAQRetrieval: 100%|██████████| 2/2 [16:36<00:00, 498.15s/it]
